# Xiangqi AI — CCPD Bronze → Silver

Notebook riêng dành cho **PGN CCPD**. Đọc dữ liệu gốc đã ingest từ `MyDrive/XiangqiAI/data/01_bronze/ccpd`, chạy `CCPDPipeline` bằng GameEngine trong backend và xuất `games.jsonl`, `transitions.jsonl`, `duplicates.jsonl`, `quarantine.jsonl`, `report.json`.

- Không tự tạo thêm ván cờ và không thay đổi Bronze.
- Mặc định **sample 20 PGN** (ngẫu nhiên cố định, seed 42). Sau khi xem quarantine, đổi `MODE = "full"` để xử lý đủ 595 PGN.
- Ký pháp tiếng Trung được giải mã theo Big5/UTF-8 và kiểm tra trực tiếp bằng các nước hợp lệ của GameEngine.
- PGN `1-0`/ `0-1` là **kết quả được ghi trong biên bản**, không tự hiểu là chiếu bí; `recorded_game_end` khác `terminated`.
- Các PGN không thể parse/replay vào quarantine; dữ liệu Silver chỉ chứa ván được chấp nhận, không chứa các ván sai.

## 1. Mount Drive và cấu hình

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

BRONZE_CCPD = Path("/content/drive/MyDrive/XiangqiAI/data/01_bronze/ccpd")
SILVER_DRIVE = Path("/content/drive/MyDrive/XiangqiAI/data/02_silver")
REPO_DIR = Path("/content/ML-Advance-Group")
BACKEND = REPO_DIR / "project" / "backend"
LOCAL_RUNS = Path("/content/xiangqi_ccpd_silver")

MODE = "sample"           # "sample" or "full"
SAMPLE_SIZE = 20
SEED = 42
EXPECTED_PGN = 595
BRANCH = "main"

if MODE not in {"sample", "full"} or SAMPLE_SIZE < 1:
    raise ValueError("Invalid MODE/SAMPLE_SIZE")
if not BRONZE_CCPD.is_dir():
    raise FileNotFoundError(f"Bronze CCPD not found: {BRONZE_CCPD}")
print("Bronze:", BRONZE_CCPD)

## 2. Clone backend từ main

In [ ]:
import subprocess
import sys

if not REPO_DIR.exists():
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", BRANCH,
        "https://github.com/Ko4rut/ML-Advance-Group.git", str(REPO_DIR)
    ], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "switch", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", BRANCH], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(BACKEND)], check=True)
assert (BACKEND / "scripts/build_ccpd_silver.py").is_file(), "CCPD Silver CLI missing on main"
REVISION = subprocess.check_output([
    "git", "-C", str(REPO_DIR), "rev-parse", "HEAD"
], text=True).strip()
print("Backend revision:", REVISION)

## 3. Liệt kê PGN và chọn sample/full

Không xử lý files manifest như ván cờ. `full` yêu cầu đủ 595 PGN của tập đã chọn. Nếu chưa đủ, hoàn tất Bronze Ingestion trước.

In [ ]:
import random

all_pgn = sorted(BRONZE_CCPD.glob("*.pgn"))
if not all_pgn:
    raise ValueError("No CCPD PGN files found in Bronze")
if MODE == "full" and len(all_pgn) != EXPECTED_PGN:
    raise ValueError(
        f"Expected {EXPECTED_PGN} PGN for full run; found {len(all_pgn)}. "
        "Complete Bronze ingestion before running full."
    )
if MODE == "sample":
    selected = sorted(random.Random(SEED).sample(
        all_pgn, min(SAMPLE_SIZE, len(all_pgn))
    ))
else:
    selected = all_pgn

missing_manifests = [
    p.name for p in selected
    if not p.with_name(p.name + ".manifest.json").is_file()
]
if missing_manifests:
    raise FileNotFoundError(
        "Missing Bronze manifests (sample): " + ", ".join(missing_manifests[:10])
    )
print("PGN on Drive:", len(all_pgn), "| selected:", len(selected))
print("Sample names:", [x.name for x in selected[:8]])

## 4. Stage bản sao về /content

Xử lý trên ổ tạm Colab để giảm I/O trực tiếp lên Drive. Kiểm tra checksum file nguồn và sidecar trong CLI; bản Bronze không bị thay đổi.

In [ ]:
import hashlib
import shutil
from datetime import datetime, timezone

RUN_ID = "ccpd_" + MODE + "_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
LOCAL_INPUT = LOCAL_RUNS / RUN_ID / "bronze"
LOCAL_OUTPUT = LOCAL_RUNS / RUN_ID / "silver"
LOCAL_INPUT.mkdir(parents=True, exist_ok=False)

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

for source in selected:
    local_file = LOCAL_INPUT / source.name
    shutil.copy2(source, local_file)
    shutil.copy2(source.with_name(source.name + ".manifest.json"),
                 local_file.with_name(local_file.name + ".manifest.json"))
    if sha256_file(source) != sha256_file(local_file):
        raise IOError(f"Staging checksum mismatch: {source.name}")
print("Local stage:", LOCAL_INPUT)

## 5. Chạy CCPD Silver CLI và đọc báo cáo

Engine kiểm tra tính hợp lệ từng nước cờ, deduplicate theo toàn bộ chuỗi nước đi. Quarantine không làm dừng các ván còn lại.

In [ ]:
import json

cmd = [
    sys.executable, "-m", "scripts.build_ccpd_silver",
    str(LOCAL_INPUT), "--output-dir", str(LOCAL_OUTPUT),
]
process = subprocess.run(cmd, cwd=BACKEND, capture_output=True, text=True)
if process.returncode != 0:
    print(process.stdout)
    print(process.stderr)
    raise RuntimeError(f"CCPD Silver pipeline failed: {process.returncode}")

report = json.loads(process.stdout)
assert report["source_files"] == len(selected)
assert report["source_files"] == (
    report["accepted_games"] + report["duplicate_games"] +
    report["quarantined_records"]
)
print(json.dumps(report, ensure_ascii=False, indent=2))
print("Accepted percentage:", round(
    100 * report["accepted_games"] / report["source_files"], 2
), "%")
if report["quarantined_records"]:
    print("Inspect quarantine.jsonl before training.")

## 6. Đồng bộ Silver lên Drive và xác nhận SHA-256

Kết quả có run ID mới, không ghi đè. Dùng thư mục `.uploading` cho đến khi toàn bộ file qua kiểm tra SHA-256.

In [ ]:
SILVER_DRIVE.mkdir(parents=True, exist_ok=True)
FINAL = SILVER_DRIVE / RUN_ID
TEMP = SILVER_DRIVE / ("." + RUN_ID + ".uploading")
if FINAL.exists() or TEMP.exists():
    raise FileExistsError("Silver run already exists; choose new RUN_ID")

shutil.copytree(LOCAL_OUTPUT, TEMP)
for source in LOCAL_OUTPUT.iterdir():
    if not source.is_file():
        continue
    if sha256_file(source) != sha256_file(TEMP / source.name):
        raise IOError(f"Silver copy checksum mismatch: {source.name}")
TEMP.rename(FINAL)

print("Verified and saved:", FINAL)
print("Silver files:", sorted(p.name for p in FINAL.iterdir()))

## Sau khi chạy

Nếu mẫu 20 PGN có kết quả hợp lệ và quarantine được giải thích, đổi `MODE = "full"` rồi chạy lại toàn bộ notebook. PGN lỗi phải được xem trong `quarantine.jsonl` và sửa parser/adapter, không sửa Bronze hay ép dữ liệu sai qua Silver.

Bộ dữ liệu này chứa cờ tướng người chơi (`source_type="human"`). Giá trị `recorded_reward` hiện là null; reward và train/validation/test split phải được xử lý ở Gold, không được tự gán trong Silver.